# US Accidents — Stratified Sampling
---
**File:** `01_sampling.ipynb`
**Purpose:** Reduce the full 7.7M-row US Accidents dataset to a manageable ~1M-row stratified sample.
**Input:** `../data/US_Accidents_March23.csv` (full dataset, not included in repo — too large)
**Output:** `../data/US_Accidents_1M.csv` (used by all subsequent notebooks)
**Run this notebook:** Only once, unless regenerating the sample with a different seed/size.
---

This notebook creates a ~1,000,000 record stratified sample (by Year × Severity) from the full US Accidents (2016–2023) dataset, using memory-efficient chunked reading suitable for large files.

In [1]:
import os
import numpy as np
import pandas as pd

INPUT_FILE = os.path.expanduser("../data/US_Accidents_March23.csv")
OUTPUT_FILE = "../data/US_Accidents_1M.csv"
TARGET_ROWS = 1_000_000 #target number of rows to sample
CHUNK_SIZE = 100_000  #read file in chunks to avoid loading 7.7M rows into memory at once
RANDOM_SEED = 42

## 1. Calculate Year × Severity Distribution
Reads the full file in chunks (memory-efficient) to compute the sampling fraction needed to reach the target sample size while preserving the original Year × Severity proportions.

In [2]:
print("Calculating Year x Severity distribution...")

group_counts = {}

for chunk in pd.read_csv(INPUT_FILE, usecols=["Start_Time", "Severity"], chunksize=CHUNK_SIZE):

    chunk["Year"] = pd.to_datetime(
        chunk["Start_Time"].astype(str).str.strip(),
        format="mixed",
        errors="coerce"
    ).dt.year

    counts = chunk.groupby(["Year", "Severity"]).size()

    for group, count in counts.items():
        group_counts[group] = group_counts.get(group, 0) + count

Calculating Year x Severity distribution...


In [3]:
#Compute the sampling fraction and display the distribution
group_counts = pd.Series(group_counts).sort_index()
total_rows = int(group_counts.sum())
sampling_fraction = TARGET_ROWS / total_rows

print(f"\nOriginal records: {total_rows:,}")
print("\nYear x Severity distribution:")
print(group_counts)
print(f"\nSampling fraction: {sampling_fraction:.6f}")


Original records: 7,728,394

Year x Severity distribution:
2016  1        225
      2     270026
      3     126215
      4      14355
2017  1        282
      2     462474
      3     231169
      4      24168
2018  1        259
      2     574668
      3     292706
      4      25793
2019  1        203
      2     688144
      3     237853
      4      28103
2020  1      28371
      2     934123
      3     185352
      4      31067
2021  1        151
      2    1384628
      3     148085
      4      30889
2022  1      37875
      2    1603346
      3      77957
      4      43274
2023  2     239572
      4       7061
dtype: int64

Sampling fraction: 0.129393


## 2. Create the Stratified Sample
Applies the computed sampling fraction within each Year × Severity group, chunk by chunk, writing results incrementally to disk rather than loading the full 7.7M-row file into memory at once.

In [4]:
#Set up the sampling loop
print("Creating stratified sample...")

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

rng = np.random.default_rng(RANDOM_SEED)
first_write = True
sampled_rows = 0

Creating stratified sample...


In [5]:
#Sample each chunk and write to disk
for chunk_number, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE), start=1):

    chunk["Year"] = pd.to_datetime(
        chunk["Start_Time"].astype(str).str.strip(),
        format="mixed",
        errors="coerce"
    ).dt.year

    selected = np.zeros(len(chunk), dtype=bool)

    for _, group in chunk.groupby(["Year", "Severity"], sort=False):
        random_values = rng.random(len(group))
        mask = random_values < sampling_fraction
        selected[group.index - chunk.index[0]] = mask

    sampled_chunk = chunk.loc[selected].copy()
    sampled_chunk.drop(columns=["Year"], inplace=True)

    if len(sampled_chunk) > 0:
        sampled_chunk.to_csv(OUTPUT_FILE, mode="w" if first_write else "a", header=first_write, index=False)
        first_write = False
        sampled_rows += len(sampled_chunk)

    if chunk_number % 10 == 0:
        print(f"Processed {chunk_number} chunks | Sampled: {sampled_rows:,}")

Processed 10 chunks | Sampled: 129,360
Processed 20 chunks | Sampled: 258,537
Processed 30 chunks | Sampled: 388,165
Processed 40 chunks | Sampled: 517,873
Processed 50 chunks | Sampled: 646,816
Processed 60 chunks | Sampled: 775,917
Processed 70 chunks | Sampled: 905,524


### Summary

In [6]:
print("\n" + "=" * 70)
print("SAMPLING COMPLETE")
print("=" * 70)

print(f"Original records : {total_rows:,}")
print(f"Sampled records  : {sampled_rows:,}")
print(f"Output file      : {OUTPUT_FILE}")
print(f"Random seed      : {RANDOM_SEED}")
print("=" * 70)


SAMPLING COMPLETE
Original records : 7,728,394
Sampled records  : 999,592
Output file      : ../data/US_Accidents_1M.csv
Random seed      : 42
